In [9]:
import os
import torch
import re
from collections import Counter
from torchvision.datasets import CocoCaptions
import torchvision.transforms as transforms



TRAIN_FILE = "/home/jovyan/megan/labb3/coco_train_preprocessed.pt"

# Where we will save the new, fixed validation file
VAL_SAVE_PATH = "/home/jovyan/lab3/coco_val_preprocessed_new.pt"

# COCO Data source paths
VAL_IMAGE_DIR = "/home/jovyan/megan/labb3/data/coco/val2017"
VAL_ANNOTATION_FILE = "/home/jovyan/megan/labb3/data/coco/annotations/captions_val2017.json"



if not os.path.exists(TRAIN_FILE):
    raise FileNotFoundError(f"Missing {TRAIN_FILE}. Run your training script first.")

print(f"Loading reference vocabulary from: {TRAIN_FILE}")
train_data = torch.load(TRAIN_FILE, weights_only=False)

vocab = train_data["vocab"]
idx_to_word = train_data["idx_to_word"]
max_length = train_data.get("max_length", 30)

print(f"Vocab loaded successfully. Size: {len(vocab)}")


def clean_caption(caption):
    caption = caption.lower()
    caption = re.sub(r"[^a-z0-9\s]", "", caption)
    caption = re.sub(r"\s+", " ", caption).strip()
    return caption

def tokenize_caption(caption, vocab, max_length=30):
    """Converts caption to IDs using the LOADED vocab."""
    cleaned = clean_caption(caption)
    tokens = ["<start>"] + cleaned.split() + ["<end>"]
    
 
    token_ids = [vocab.get(token, vocab["<unk>"]) for token in tokens]
    
    # Padding/Truncating
    if len(token_ids) < max_length:
        token_ids += [vocab["<pad>"]] * (max_length - len(token_ids))
    else:
        token_ids = token_ids[:max_length]
        
    return torch.tensor(token_ids, dtype=torch.long)


print("Initializing COCO Validation Dataset...")
transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])
coco_dataset = CocoCaptions(root=VAL_IMAGE_DIR, annFile=VAL_ANNOTATION_FILE, transform=transform)

def create_valid_val_set():
    samples = []
    print(f"Preprocessing {len(coco_dataset)} images...")

    for i in range(len(coco_dataset)):
        image_id = coco_dataset.ids[i]
        image_info = coco_dataset.coco.loadImgs(image_id)[0]
        filename = image_info["file_name"]
        image_path = os.path.join(VAL_IMAGE_DIR, filename)

        _, captions = coco_dataset[i]

        for caption in captions:
            token_ids = tokenize_caption(caption, vocab, max_length=max_length)
            samples.append({
                "image_path": image_path,
                "caption": caption,
                "token_ids": token_ids
            })

        if i % 1000 == 0 and i > 0:
            print(f"Processed {i}/{len(coco_dataset)} images")


    output_data = {
        "samples": samples,
        "vocab": vocab,           # Exact same as training
        "idx_to_word": idx_to_word, # Exact same as training
        "max_length": max_length
    }

    # Ensure directory exists before saving
    os.makedirs(os.path.dirname(VAL_SAVE_PATH), exist_ok=True)
    
    torch.save(output_data, VAL_SAVE_PATH)
    print(f"\n--- SUCCESS ---")
    print(f"Saved {len(samples)} samples to: {VAL_SAVE_PATH}")

if __name__ == "__main__":
    create_valid_val_set()

Loading reference vocabulary from: /home/jovyan/megan/labb3/coco_train_preprocessed.pt
Vocab loaded successfully. Size: 10307
Initializing COCO Validation Dataset...
loading annotations into memory...
Done (t=0.05s)
creating index...
index created!
Preprocessing 5000 images...
Processed 1000/5000 images
Processed 2000/5000 images
Processed 3000/5000 images
Processed 4000/5000 images

--- SUCCESS ---
Saved 25014 samples to: /home/jovyan/lab3/coco_val_preprocessed_new.pt
